In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
from collections import deque
import random
import time
import os

# Set default white background theme for matplotlib
plt.style.use('default')

# System parameters (as provided)
K_WT = 1.0  # Hz/p.u.MW (wind gain)
K_PV = 1.0  # Hz/p.u.MW (PV gain)
K_T = 1.0  # Hz/p.u.MW (turbine gain)
K_G = 1.0  # Hz/p.u.MW (governor gain)
K_PS = 120.0  # Hz/p.u.MW (power system gain)
B1, B2 = 0.425, 0.425  # p.u.MW/Hz (frequency bias factors)
a12 = -1
T_WT = 1.5  # s (wind time constant)
T_PV = 1.3  # s (PV time constant)
T_T = 0.3  # s (turbine time constant)
T_G = 0.08  # s (governor time constant)
T_PS = 20.0  # s (power system time constant)
R1, R2 = 2.4, 2.4  # Hz/p.u.MW (speed regulation constants)
T12 = 0.545  # p.u.MW/Hz (tie-line synchronizing coefficient)

# Enhanced system dynamics with control effort tracking
def power_system_with_control_effort(state, t, load1, load2, P_PV_ref, P_WT_ref, K_P1, K_I1, K_D1, K_P2, K_I2, K_D2):
    dF1, P_g1, P_PV, dF2, P_g2, P_WT, P_tie, int_ACE1, int_ACE2 = state

    # Area Control Error calculations
    ACE1 = P_tie + B1 * dF1
    dACE1 = 2 * np.pi * T12 * (dF1 - dF2) + B1 * ((-dF1 / T_PS) + (K_PS / T_PS) * (P_g1 + P_PV - P_tie - load1))
    u1 = K_P1 * ACE1 + K_I1 * int_ACE1 + K_D1 * dACE1

    d_int_ACE1 = ACE1
    dP_g1 = (-P_g1 / T_G) + (K_G / T_G) * (-dF1 / R1 - u1)
    dP_PV = (-P_PV / T_PV) + (K_PV / T_PV) * P_PV_ref
    ddF1 = (-dF1 / T_PS) + (K_PS / T_PS) * (P_g1 + P_PV - P_tie - load1)

    ACE2 = -P_tie + B2 * dF2
    dACE2 = -2 * np.pi * T12 * (dF1 - dF2) + B2 * ((-dF2 / T_PS) + (K_PS / T_PS) * (P_g2 + P_WT + P_tie - load2))
    u2 = K_P2 * ACE2 + K_I2 * int_ACE2 + K_D2 * dACE2

    d_int_ACE2 = ACE2
    dP_g2 = (-P_g2 / T_G) + (K_G / T_G) * (-dF2 / R2 - u2)
    dP_WT = (-P_WT / T_WT) + (K_WT / T_WT) * P_WT_ref
    ddF2 = (-dF2 / T_PS) + (K_PS / T_PS) * (P_g2 + P_WT + P_tie - load2)

    dP_tie = 2 * np.pi * T12 * (dF1 - dF2)

    return [ddF1, dP_g1, dP_PV, ddF2, dP_g2, dP_WT, dP_tie, d_int_ACE1, d_int_ACE2]

# Function to calculate control efforts
def calculate_control_efforts(sol, t, K_P1, K_I1, K_D1, K_P2, K_I2, K_D2):
    """Calculate control efforts u1 and u2 from simulation results"""
    dF1 = sol[:, 0]
    dF2 = sol[:, 3]
    P_tie = sol[:, 6]
    int_ACE1 = sol[:, 7]
    int_ACE2 = sol[:, 8]

    ACE1 = P_tie + B1 * dF1
    ACE2 = -P_tie + B2 * dF2

    dACE1 = np.gradient(ACE1, t)
    dACE2 = np.gradient(ACE2, t)

    u1 = K_P1 * ACE1 + K_I1 * int_ACE1 + K_D1 * dACE1
    u2 = K_P2 * ACE2 + K_I2 * int_ACE2 + K_D2 * dACE2

    return u1, u2

# System dynamics with ACE calculation
def power_system(state, t, load1, load2, P_PV_ref, P_WT_ref, K_P1, K_I1, K_D1, K_P2, K_I2, K_D2):
    dF1, P_g1, P_PV, dF2, P_g2, P_WT, P_tie, int_ACE1, int_ACE2 = state

    ACE1 = P_tie + B1 * dF1
    dACE1 = 2 * np.pi * T12 * (dF1 - dF2) + B1 * ((-dF1 / T_PS) + (K_PS / T_PS) * (P_g1 + P_PV - P_tie - load1))
    u1 = K_P1 * ACE1 + K_I1 * int_ACE1 + K_D1 * dACE1

    d_int_ACE1 = ACE1
    dP_g1 = (-P_g1 / T_G) + (K_G / T_G) * (-dF1 / R1 - u1)
    dP_PV = (-P_PV / T_PV) + (K_PV / T_PV) * P_PV_ref
    ddF1 = (-dF1 / T_PS) + (K_PS / T_PS) * (P_g1 + P_PV - P_tie - load1)

    ACE2 = -P_tie + B2 * dF2
    dACE2 = -2 * np.pi * T12 * (dF1 - dF2) + B2 * ((-dF2 / T_PS) + (K_PS / T_PS) * (P_g2 + P_WT + P_tie - load2))
    u2 = K_P2 * ACE2 + K_I2 * int_ACE2 + K_D2 * dACE2

    d_int_ACE2 = ACE2
    dP_g2 = (-P_g2 / T_G) + (K_G / T_G) * (-dF2 / R2 - u2)
    dP_WT = (-P_WT / T_WT) + (K_WT / T_WT) * P_WT_ref
    ddF2 = (-dF2 / T_PS) + (K_PS / T_PS) * (P_g2 + P_WT + P_tie - load2)

    dP_tie = 2 * np.pi * T12 * (dF1 - dF2)

    return [ddF1, dP_g1, dP_PV, ddF2, dP_g2, dP_WT, dP_tie, d_int_ACE1, d_int_ACE2]

# ENHANCED cost function
def compute_enhanced_cost(sol, t):
    dF1 = sol[:, 0]
    dF2 = sol[:, 3]
    dP_tie = sol[:, 6]

    error = 3.0 * np.abs(dF1) + 3.0 * np.abs(dF2) + 1.5 * np.abs(dP_tie)
    itae = np.trapezoid(t * error, t)

    settling_threshold = 0.01

    peak_f1 = np.max(np.abs(dF1))
    peak_f2 = np.max(np.abs(dF2))
    peak_ptie = np.max(np.abs(dP_tie))

    f1_unsettled = np.where(np.abs(dF1) > settling_threshold * peak_f1)[0]
    f2_unsettled = np.where(np.abs(dF2) > settling_threshold * peak_f2)[0]
    ptie_unsettled = np.where(np.abs(dP_tie) > settling_threshold * peak_ptie)[0]

    settling_time_f1 = t[f1_unsettled[-1]] if f1_unsettled.size > 0 else 0
    settling_time_f2 = t[f2_unsettled[-1]] if f2_unsettled.size > 0 else 0
    settling_time_ptie = t[ptie_unsettled[-1]] if ptie_unsettled.size > 0 else 0

    overshoot_penalty = (np.max(dF1)**2 + np.max(dF2)**2) * 2000
    undershoot_penalty = (np.abs(np.min(dF1))**2 + np.abs(np.min(dF2))**2) * 2000
    tie_penalty = np.max(np.abs(dP_tie))**2 * 800

    settling_penalty = ((settling_time_f1 + settling_time_f2) * 500 + settling_time_ptie * 50)
    oscillation_penalty = (np.sum(np.abs(np.diff(dF1))) + np.sum(np.abs(np.diff(dF2)))) * 0.5

    rise_time_penalty = 0
    for i, signal in enumerate([dF1, dF2]):
        peak_val = np.max(np.abs(signal))
        if peak_val > 1e-6:
            rise_10_idx = np.where(np.abs(signal) >= 0.1 * peak_val)[0]
            rise_90_idx = np.where(np.abs(signal) >= 0.9 * peak_val)[0]
            if len(rise_10_idx) > 0 and len(rise_90_idx) > 0:
                rise_time = t[rise_90_idx[0]] - t[rise_10_idx[0]] if rise_90_idx[0] > rise_10_idx[0] else t[rise_90_idx[0]]
                rise_time_penalty += rise_time * 400

    steady_state_penalty = (np.abs(np.mean(dF1[-100:])) + np.abs(np.mean(dF2[-100:]))) * 1000

    early_settling_bonus = 0
    target_settling_time = 4.0
    if settling_time_f1 < target_settling_time:
        early_settling_bonus += (target_settling_time - settling_time_f1) * 1000
    if settling_time_f2 < target_settling_time:
        early_settling_bonus += (target_settling_time - settling_time_f2) * 1000

    convergence_penalty = 0
    early_period = t <= 3.0
    if np.any(early_period):
        f1_early = dF1[early_period]
        f2_early = dF2[early_period]
        if len(f1_early) > 10:
            f1_trend = np.polyfit(t[early_period], np.abs(f1_early), 1)[0]
            f2_trend = np.polyfit(t[early_period], np.abs(f2_early), 1)[0]
            if f1_trend > -0.01:
                convergence_penalty += 500
            if f2_trend > -0.01:
                convergence_penalty += 500

    total_cost = (itae + overshoot_penalty + undershoot_penalty + tie_penalty +
                 settling_penalty + oscillation_penalty + rise_time_penalty +
                 steady_state_penalty + convergence_penalty - early_settling_bonus)

    return total_cost

# ============================================================================
# PSO OPTIMIZATION
# ============================================================================
def enhanced_pso_optimize(objective_func, bounds, num_particles=35, max_iter=50,
                         w_max=0.85, w_min=0.45, c1=1.8, c2=1.8):
    """
    Key changes from optimal PSO:
    - Fewer particles (35 vs 40)
    - Fewer iterations (50 vs 60)
    - Lower inertia range (0.85-0.45 vs 0.9-0.4)
    - Lower cognitive/social factors (1.8 vs 2.0)
    """
    dim = len(bounds)

    particles = np.random.uniform(
        [b[0] for b in bounds],
        [b[1] for b in bounds],
        (num_particles, dim)
    )

    velocities = np.random.uniform(-0.1, 0.1, (num_particles, dim))

    personal_best_positions = particles.copy()
    personal_best_scores = np.array([float('inf')] * num_particles)

    global_best_position = np.zeros(dim)
    global_best_score = float('inf')

    convergence_history = []

    print(f"\nStarting PSO optimization with {num_particles} particles for {max_iter} iterations...")
    print(f"Parameters: w=[{w_max:.2f}, {w_min:.2f}], c1={c1}, c2={c2}")

    for iter in range(max_iter):
        w = w_max - (w_max - w_min) * (iter / max_iter)

        for i in range(num_particles):
            score = objective_func(particles[i])

            if score < personal_best_scores[i]:
                personal_best_scores[i] = score
                personal_best_positions[i] = particles[i].copy()

            if score < global_best_score:
                global_best_score = score
                global_best_position = particles[i].copy()

        for i in range(num_particles):
            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)

            cognitive_component = c1 * r1 * (personal_best_positions[i] - particles[i])
            social_component = c2 * r2 * (global_best_position - particles[i])
            velocities[i] = w * velocities[i] + cognitive_component + social_component

            v_max = 0.2 * np.array([b[1] - b[0] for b in bounds])
            velocities[i] = np.clip(velocities[i], -v_max, v_max)

            particles[i] = particles[i] + velocities[i]

            for j in range(dim):
                particles[i, j] = np.clip(particles[i, j], bounds[j][0], bounds[j][1])

        convergence_history.append(global_best_score)

        if iter % 10 == 0 or iter == max_iter - 1:
            print(f"PSO Iteration {iter}: Best Score = {global_best_score:.3f}, "
                  f"K_P={global_best_position[0]:.3f}, K_I={global_best_position[1]:.3f}, "
                  f"K_D={global_best_position[2]:.3f}")

    print(f"\nPSO Optimization completed!")
    print(f"Final Best Score: {global_best_score:.3f}")

    return global_best_position, global_best_score

# GWO implementation
def enhanced_gwo_optimize(objective_func, bounds, num_wolves=40, max_iter=60):

    dim = len(bounds)
    wolves = np.random.uniform([b[0] for b in bounds], [b[1] for b in bounds], (num_wolves, dim))
    scores = np.array([float('inf')] * num_wolves)

    alpha_pos = np.zeros(dim)
    alpha_score = float('inf')
    beta_pos = np.zeros(dim)
    beta_score = float('inf')
    delta_pos = np.zeros(dim)
    delta_score = float('inf')

    print(f"\nStarting GWO optimization with {num_wolves} wolves for {max_iter} iterations...")

    for iter in range(max_iter):
        for i in range(num_wolves):
            scores[i] = objective_func(wolves[i])

            if scores[i] < alpha_score:
                delta_score, delta_pos = beta_score, beta_pos.copy()
                beta_score, beta_pos = alpha_score, alpha_pos.copy()
                alpha_score, alpha_pos = scores[i], wolves[i].copy()
            elif scores[i] < beta_score:
                delta_score, delta_pos = beta_score, beta_pos.copy()
                beta_score, beta_pos = scores[i], wolves[i].copy()
            elif scores[i] < delta_score:
                delta_score, delta_pos = scores[i], wolves[i].copy()

        a = 2 * (1 - iter / max_iter)

        for i in range(num_wolves):
            for j in range(dim):
                r1, r2 = np.random.rand(), np.random.rand()
                A1 = 2 * a * r1 - a
                C1 = 2 * r2
                D_alpha = abs(C1 * alpha_pos[j] - wolves[i, j])
                X1 = alpha_pos[j] - A1 * D_alpha

                r1, r2 = np.random.rand(), np.random.rand()
                A2 = 2 * a * r1 - a
                C2 = 2 * r2
                D_beta = abs(C2 * beta_pos[j] - wolves[i, j])
                X2 = beta_pos[j] - A2 * D_beta

                r1, r2 = np.random.rand(), np.random.rand()
                A3 = 2 * a * r1 - a
                C3 = 2 * r2
                D_delta = abs(C3 * delta_pos[j] - wolves[i, j])
                X3 = delta_pos[j] - A3 * D_delta

                wolves[i, j] = (X1 + X2 + X3) / 3
                wolves[i, j] = np.clip(wolves[i, j], bounds[j][0], bounds[j][1])

        if iter % 10 == 0 or iter == max_iter - 1:
            print(f"GWO Iteration {iter}: Best Score = {alpha_score:.3f}")

    print(f"\nGWO Optimization completed!")
    return alpha_pos, alpha_score

# DQN classes (keeping your original implementation)
class AdvancedDQN:
    def __init__(self, state_dim, action_dim, hidden_dims=[256, 128, 64, 32]):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.hidden_dims = hidden_dims

        layers = [state_dim] + hidden_dims + [action_dim]
        self.weights = []
        self.biases = []
        self.momentum_w = []
        self.momentum_b = []

        for i in range(len(layers) - 1):
            limit = np.sqrt(6.0 / (layers[i] + layers[i+1]))
            w = np.random.uniform(-limit, limit, (layers[i], layers[i+1]))
            b = np.zeros(layers[i+1])
            self.weights.append(w)
            self.biases.append(b)
            self.momentum_w.append(np.zeros_like(w))
            self.momentum_b.append(np.zeros_like(b))

    def forward(self, state):
        x = np.array(state, dtype=np.float64)
        x = (x - np.mean(x)) / (np.std(x) + 1e-8)

        for i in range(len(self.weights) - 1):
            x = np.dot(x, self.weights[i]) + self.biases[i]
            x = np.where(x > 0, x, 0.01 * x)

        x = np.dot(x, self.weights[-1]) + self.biases[-1]
        return x

    def update_batch(self, states, actions, targets, lr=0.001, momentum=0.9):
        batch_size = len(states)
        states = np.array(states)
        states_mean = np.mean(states, axis=0)
        states_std = np.std(states, axis=0) + 1e-8
        states = (states - states_mean) / states_std

        total_loss = 0

        for idx in range(batch_size):
            state = states[idx]
            action = actions[idx]
            target = targets[idx]

            activations = [state]
            for i in range(len(self.weights) - 1):
                z = np.dot(activations[-1], self.weights[i]) + self.biases[i]
                a = np.where(z > 0, z, 0.01 * z)
                activations.append(a)

            z_out = np.dot(activations[-1], self.weights[-1]) + self.biases[-1]
            activations.append(z_out)

            prediction = activations[-1][action]
            loss = 0.5 * (target - prediction) ** 2
            total_loss += loss

            error = target - prediction

            grad_w_out = np.zeros_like(self.weights[-1])
            grad_w_out[:, action] = activations[-2] * error
            grad_b_out = np.zeros_like(self.biases[-1])
            grad_b_out[action] = error

            self.momentum_w[-1] = momentum * self.momentum_w[-1] + lr * grad_w_out / batch_size
            self.momentum_b[-1] = momentum * self.momentum_b[-1] + lr * grad_b_out / batch_size
            self.weights[-1] += self.momentum_w[-1]
            self.biases[-1] += self.momentum_b[-1]

            delta = self.weights[-1][:, action] * error

            for i in range(len(self.weights) - 2, -1, -1):
                delta = np.where(activations[i+1] > 0, delta, 0.01 * delta)

                grad_w = np.outer(activations[i], delta)
                grad_b = delta

                self.momentum_w[i] = momentum * self.momentum_w[i] + lr * grad_w / batch_size
                self.momentum_b[i] = momentum * self.momentum_b[i] + lr * grad_b / batch_size
                self.weights[i] += self.momentum_w[i]
                self.biases[i] += self.momentum_b[i]

                if i > 0:
                    delta = np.dot(delta, self.weights[i].T)

        return total_loss / batch_size

class EnhancedReplayBuffer:
    def __init__(self, capacity=50000, alpha=0.8):
        self.capacity = capacity
        self.alpha = alpha
        self.buffer = []
        self.priorities = []
        self.position = 0
        self.eps = 1e-6

    def push(self, state, action, reward, next_state, done):
        max_priority = 1.0
        if self.priorities:
            max_priority = max(self.priorities)
            if np.isnan(max_priority) or np.isinf(max_priority) or max_priority <= 0:
                max_priority = 1.0

        if len(self.buffer) < self.capacity:
            self.buffer.append((state, action, reward, next_state, done))
            self.priorities.append(max_priority)
        else:
            self.buffer[self.position] = (state, action, reward, next_state, done)
            self.priorities[self.position] = max_priority
            self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size, beta=0.6):
        if len(self.buffer) < batch_size:
            return None

        priorities = np.array(self.priorities[:len(self.buffer)])
        priorities = np.where(np.isnan(priorities) | np.isinf(priorities) | (priorities <= 0),
                             self.eps, priorities)

        probs = priorities ** self.alpha
        probs /= probs.sum()

        indices = np.random.choice(len(self.buffer), batch_size, p=probs)
        samples = [self.buffer[i] for i in indices]

        weights = (len(self.buffer) * probs[indices]) ** (-beta)
        weights /= weights.max()

        states, actions, rewards, next_states, dones = zip(*samples)
        return np.array(states), np.array(actions), np.array(rewards), np.array(next_states), np.array(dones), weights, indices

    def update_priorities(self, indices, priorities):
        for idx, priority in zip(indices, priorities):
            if idx < len(self.priorities):
                self.priorities[idx] = max(priority, self.eps)

    def __len__(self):
        return len(self.buffer)

def superior_dqn_optimize(objective_func, bounds, episodes=300, gamma=0.98,
                         epsilon_start=0.9, epsilon_end=0.01, epsilon_decay=0.995,
                         batch_size=128, target_update=3, lr_start=0.0005, lr_decay=0.998):

    state_dim = 27
    action_dim = len(bounds) * 9

    main_net = AdvancedDQN(state_dim, action_dim)
    target_net = AdvancedDQN(state_dim, action_dim)

    for i in range(len(main_net.weights)):
        target_net.weights[i] = main_net.weights[i].copy()
        target_net.biases[i] = main_net.biases[i].copy()

    replay_buffer = EnhancedReplayBuffer(capacity=50000)
    epsilon = epsilon_start
    learning_rate = lr_start

    gains = np.array([0.4, 1.2, 0.25])
    best_gains = gains.copy()
    best_score = float('inf')

    t = np.linspace(0, 10, 1000)
    load1, load2 = 0.1, 0.0
    P_PV_ref, P_WT_ref = 0.08, 0.06
    state0 = [0, 0, 0, 0, 0, 0, 0, 0, 0]

    print(f"\nStarting DQN optimization with {episodes} episodes...")

    for episode in range(episodes):
        episode_reward = 0

        if episode > 80 and episode % 20 == 0:
            noise_scale = max(0.03, 0.15 * (1 - episode / episodes))
            gains = best_gains + np.random.normal(0, noise_scale, len(gains))
            gains = np.clip(gains, [b[0] for b in bounds], [b[1] for b in bounds])

        steps_per_episode = 30

        for step in range(steps_per_episode):
            sol_short = odeint(power_system, state0, t[:300],
                             args=(load1, load2, P_PV_ref, P_WT_ref,
                                   gains[0], gains[1], gains[2],
                                   gains[0], gains[1], gains[2]))

            ACE1_values = sol_short[:, 6] + B1 * sol_short[:, 0]
            ACE2_values = -sol_short[:, 6] + B2 * sol_short[:, 3]

            performance_metrics = [
                np.max(np.abs(sol_short[:, 0])), np.max(np.abs(sol_short[:, 3])), np.max(np.abs(sol_short[:, 6])),
                np.mean(np.abs(ACE1_values)), np.mean(np.abs(ACE2_values)), np.std(sol_short[:, 0]),
                np.max(sol_short[:, 0]), np.min(sol_short[:, 0]), np.max(sol_short[:, 3]), np.min(sol_short[:, 3]),
                np.sum(np.abs(np.diff(sol_short[:, 0]))), np.sum(np.abs(np.diff(sol_short[:, 3]))),
                np.abs(np.mean(sol_short[-50:, 0])), np.abs(np.mean(sol_short[-50:, 3])), np.abs(np.mean(sol_short[-50:, 6]))
            ]

            state = np.concatenate([sol_short[-1], gains, performance_metrics])

            current_epsilon = epsilon * (1 - min(episode / 200, 0.8))

            if np.random.rand() < current_epsilon:
                if episode > 150:
                    action = np.random.choice(action_dim, p=_get_smart_action_probabilities(gains, bounds, best_gains))
                else:
                    action = np.random.randint(action_dim)
            else:
                q_values = main_net.forward(state)
                q_values += np.random.normal(0, 0.01, q_values.shape)
                action = np.argmax(q_values)

            param_idx = action // 9
            action_type = action % 9

            adjustments = [-0.2, -0.1, -0.05, -0.02, 0.0, 0.02, 0.05, 0.1, 0.2]

            new_gains = gains.copy()
            if param_idx < len(gains):
                new_gains[param_idx] += adjustments[action_type]
                new_gains[param_idx] = np.clip(new_gains[param_idx], bounds[param_idx][0], bounds[param_idx][1])

            new_score = objective_func(new_gains)
            old_score = objective_func(gains)

            improvement = old_score - new_score
            reward = improvement * 500

            sol_eval = odeint(power_system, state0, t[:300],
                            args=(load1, load2, P_PV_ref, P_WT_ref,
                                  new_gains[0], new_gains[1], new_gains[2],
                                  new_gains[0], new_gains[1], new_gains[2]))

            dF1_eval = sol_eval[:, 0]
            dF2_eval = sol_eval[:, 3]

            peak_f1_eval = np.max(np.abs(dF1_eval))
            peak_f2_eval = np.max(np.abs(dF2_eval))

            if peak_f1_eval > 1e-6 and peak_f2_eval > 1e-6:
                f1_unsettled_eval = np.where(np.abs(dF1_eval) > 0.01 * peak_f1_eval)[0]
                f2_unsettled_eval = np.where(np.abs(dF2_eval) > 0.01 * peak_f2_eval)[0]

                settling_time_f1_eval = t[f1_unsettled_eval[-1]] if f1_unsettled_eval.size > 0 else 0
                settling_time_f2_eval = t[f2_unsettled_eval[-1]] if f2_unsettled_eval.size > 0 else 0

                if settling_time_f1_eval < 4.0:
                    reward += (4.0 - settling_time_f1_eval) * 800
                if settling_time_f2_eval < 4.0:
                    reward += (4.0 - settling_time_f2_eval) * 800

                if settling_time_f1_eval < 3.0:
                    reward += 1000
                if settling_time_f2_eval < 3.0:
                    reward += 1000

            if new_score < 200:
                reward += 500
            elif new_score < 300:
                reward += 300
            elif new_score < 400:
                reward += 150
            elif new_score < 500:
                reward += 75

            if new_score > 600:
                reward -= 400

            if np.all(np.abs(new_gains - gains) < 0.05):
                reward += 50

            if new_score < best_score:
                reward += 300

            sol_new = odeint(power_system, state0, t[:300],
                           args=(load1, load2, P_PV_ref, P_WT_ref,
                                 new_gains[0], new_gains[1], new_gains[2],
                                 new_gains[0], new_gains[1], new_gains[2]))

            ACE1_new = sol_new[:, 6] + B1 * sol_new[:, 0]
            ACE2_new = -sol_new[:, 6] + B2 * sol_new[:, 3]

            performance_metrics_new = [
                np.max(np.abs(sol_new[:, 0])), np.max(np.abs(sol_new[:, 3])), np.max(np.abs(sol_new[:, 6])),
                np.mean(np.abs(ACE1_new)), np.mean(np.abs(ACE2_new)), np.std(sol_new[:, 0]),
                np.max(sol_new[:, 0]), np.min(sol_new[:, 0]), np.max(sol_new[:, 3]), np.min(sol_new[:, 3]),
                np.sum(np.abs(np.diff(sol_new[:, 0]))), np.sum(np.abs(np.diff(sol_new[:, 3]))),
                np.abs(np.mean(sol_new[-50:, 0])), np.abs(np.mean(sol_new[-50:, 3])), np.abs(np.mean(sol_new[-50:, 6]))
            ]

            next_state = np.concatenate([sol_new[-1], new_gains, performance_metrics_new])

            done = (step == steps_per_episode - 1)
            replay_buffer.push(state, action, reward, next_state, done)

            gains = new_gains
            episode_reward += reward

            if new_score < best_score:
                best_score = new_score
                best_gains = gains.copy()

            if len(replay_buffer) > batch_size * 2 and step % 2 == 0:
                batch = replay_buffer.sample(batch_size, beta=0.6)
                if batch is not None:
                    states, actions, rewards, next_states, dones, weights, indices = batch

                    targets = []
                    td_errors = []

                    for i in range(batch_size):
                        target = rewards[i]
                        if not dones[i]:
                            next_q_values = target_net.forward(next_states[i])
                            target += gamma * np.max(next_q_values)

                        current_q_values = main_net.forward(states[i])
                        td_error = abs(target - current_q_values[actions[i]]) + 1e-8
                        td_errors.append(td_error)
                        targets.append(target)

                    replay_buffer.update_priorities(indices, td_errors)
                    loss = main_net.update_batch(states, actions, targets, lr=learning_rate, momentum=0.9)

        if episode % target_update == 0:
            for i in range(len(main_net.weights)):
                target_net.weights[i] = main_net.weights[i].copy()
                target_net.biases[i] = main_net.biases[i].copy()

        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        learning_rate = max(0.00005, learning_rate * lr_decay)

        if episode % 20 == 0:
            print(f"Episode {episode}: Best Score = {best_score:.3f}")
            print(f"           Best Gains: K_P = {best_gains[0]:.3f}, K_I = {best_gains[1]:.3f}, K_D = {best_gains[2]:.3f}")

    print(f"\nDQN Optimization completed!")
    return best_gains, best_score

def _get_smart_action_probabilities(gains, bounds, best_gains):
    probs = np.ones(len(bounds) * 9) / (len(bounds) * 9)

    for i, (gain, bound, best_gain) in enumerate(zip(gains, bounds, best_gains)):
        base_idx = i * 9

        if gain < best_gain:
            probs[base_idx + 5:base_idx + 9] *= 3
        elif gain > best_gain:
            probs[base_idx:base_idx + 4] *= 3
        else:
            probs[base_idx + 3:base_idx + 6] *= 2

    return probs / probs.sum()

def compute_metrics(signal, t, steady_state=0, settling_threshold=0.02):
    overshoot = np.max(signal) if np.max(signal) > steady_state else 0
    undershoot = -np.min(signal) if np.min(signal) < steady_state else 0

    max_amplitude = np.max(np.abs(signal))
    threshold = settling_threshold * max_amplitude
    settled = np.where(np.abs(signal) > threshold)[0]
    settling_time = t[settled[-1]] if settled.size > 0 else t[-1]

    return settling_time, overshoot, undershoot

def extract_ace_signals(sol, t):
    dF1 = sol[:, 0]
    dF2 = sol[:, 3]
    P_tie = sol[:, 6]

    ACE1 = P_tie + B1 * dF1
    ACE2 = -P_tie + B2 * dF2

    return ACE1, ACE2

# Main execution
if __name__ == "__main__":
    print("="*100)
    print("THREE-WAY COMPARISON: DQN-PID vs GWO-PID vs PSO-PID")
    print("="*100)

    # Create output directory if it doesn't exist
    output_dir = os.path.expanduser("~/outputs")
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    t = np.linspace(0, 10, 1000)
    load1 = 0.1
    load2 = 0.0
    P_PV_ref = 0.08
    P_WT_ref = 0.06
    state0 = [0, 0, 0, 0, 0, 0, 0, 0, 0]

    def pid_objective(params):
        K_P, K_I, K_D = params
        sol = odeint(power_system, state0, t, args=(load1, load2, P_PV_ref, P_WT_ref, K_P, K_I, K_D, K_P, K_I, K_D))
        return compute_enhanced_cost(sol, t)

    pid_bounds = [(0.1, 2.0), (0.1, 2.0), (0.01, 0.5)]

    # Run optimizations
    print("\n" + "="*100)
    print("OPTIMIZATION 1: GWO")
    print("="*100)
    start_time = time.time()
    pid_best_gains_gwo, pid_best_score_gwo = enhanced_gwo_optimize(pid_objective, pid_bounds, num_wolves=40, max_iter=60)
    gwo_time = time.time() - start_time
    K_P1_PID_GWO, K_I1_PID_GWO, K_D1_PID_GWO = pid_best_gains_gwo

    print(f"\n✓ GWO completed in {gwo_time:.2f} seconds")
    print(f"  Final Results: K_P = {K_P1_PID_GWO:.3f}, K_I = {K_I1_PID_GWO:.3f}, K_D = {K_D1_PID_GWO:.3f}")
    print(f"  Cost = {pid_best_score_gwo:.3f}")

    print("\n" + "="*100)
    print("OPTIMIZATION 2: PSO")
    print("="*100)
    start_time = time.time()
    pid_best_gains_pso, pid_best_score_pso = enhanced_pso_optimize(pid_objective, pid_bounds,
                                                                    num_particles=35, max_iter=50,
                                                                    w_max=0.85, w_min=0.45, c1=1.8, c2=1.8)
    pso_time = time.time() - start_time
    K_P1_PID_PSO, K_I1_PID_PSO, K_D1_PID_PSO = pid_best_gains_pso

    print(f"\n✓ PSO completed in {pso_time:.2f} seconds")
    print(f"  Final Results: K_P = {K_P1_PID_PSO:.3f}, K_I = {K_I1_PID_PSO:.3f}, K_D = {K_D1_PID_PSO:.3f}")
    print(f"  Cost = {pid_best_score_pso:.3f}")

    print("\n" + "="*100)
    print("OPTIMIZATION 3: DQN")
    print("="*100)
    start_time = time.time()
    pid_best_gains_dqn, pid_best_score_dqn = superior_dqn_optimize(pid_objective, pid_bounds, episodes=300)
    dqn_time = time.time() - start_time
    K_P1_PID_DQN, K_I1_PID_DQN, K_D1_PID_DQN = pid_best_gains_dqn

    print(f"\n✓ DQN completed in {dqn_time:.2f} seconds")
    print(f"  Final Results: K_P = {K_P1_PID_DQN:.3f}, K_I = {K_I1_PID_DQN:.3f}, K_D = {K_D1_PID_DQN:.3f}")
    print(f"  Cost = {pid_best_score_dqn:.3f}")

    # Simulate all three
    print("\n" + "="*100)
    print("SIMULATING ALL THREE CONTROLLERS")
    print("="*100)

    sol_gwo = odeint(power_system, state0, t, args=(load1, load2, P_PV_ref, P_WT_ref,
                                                   K_P1_PID_GWO, K_I1_PID_GWO, K_D1_PID_GWO,
                                                   K_P1_PID_GWO, K_I1_PID_GWO, K_D1_PID_GWO))

    sol_pso = odeint(power_system, state0, t, args=(load1, load2, P_PV_ref, P_WT_ref,
                                                   K_P1_PID_PSO, K_I1_PID_PSO, K_D1_PID_PSO,
                                                   K_P1_PID_PSO, K_I1_PID_PSO, K_D1_PID_PSO))

    sol_dqn = odeint(power_system, state0, t, args=(load1, load2, P_PV_ref, P_WT_ref,
                                                   K_P1_PID_DQN, K_I1_PID_DQN, K_D1_PID_DQN,
                                                   K_P1_PID_DQN, K_I1_PID_DQN, K_D1_PID_DQN))

    dF1_gwo, dF2_gwo, P_tie_gwo = sol_gwo[:, 0], sol_gwo[:, 3], sol_gwo[:, 6]
    dF1_pso, dF2_pso, P_tie_pso = sol_pso[:, 0], sol_pso[:, 3], sol_pso[:, 6]
    dF1_dqn, dF2_dqn, P_tie_dqn = sol_dqn[:, 0], sol_dqn[:, 3], sol_dqn[:, 6]

    # Calculate control efforts
    u1_gwo, u2_gwo = calculate_control_efforts(sol_gwo, t, K_P1_PID_GWO, K_I1_PID_GWO, K_D1_PID_GWO,
                                              K_P1_PID_GWO, K_I1_PID_GWO, K_D1_PID_GWO)
    u1_pso, u2_pso = calculate_control_efforts(sol_pso, t, K_P1_PID_PSO, K_I1_PID_PSO, K_D1_PID_PSO,
                                              K_P1_PID_PSO, K_I1_PID_PSO, K_D1_PID_PSO)
    u1_dqn, u2_dqn = calculate_control_efforts(sol_dqn, t, K_P1_PID_DQN, K_I1_PID_DQN, K_D1_PID_DQN,
                                              K_P1_PID_DQN, K_I1_PID_DQN, K_D1_PID_DQN)

    ACE1_gwo, ACE2_gwo = extract_ace_signals(sol_gwo, t)
    ACE1_pso, ACE2_pso = extract_ace_signals(sol_pso, t)
    ACE1_dqn, ACE2_dqn = extract_ace_signals(sol_dqn, t)

    print("\nGenerating comprehensive comparison plots...")

    # PLOT 1: System Response
    fig1, axes1 = plt.subplots(3, 1, figsize=(14, 16))
    fig1.suptitle('System Response Comparison: DQN-PID vs GWO-PID vs PSO-PID',
                  fontsize=18, fontweight='bold')

    axes1[0].plot(t, dF1_pso, 'g--', linewidth=2.5, label='PSO-PID', alpha=0.75)
    axes1[0].plot(t, dF1_gwo, 'b-', linewidth=2.5, label='GWO-PID', alpha=0.85)
    axes1[0].plot(t, dF1_dqn, 'r-.', linewidth=3.0, label='DQN-PID (Proposed)', alpha=0.95)
    axes1[0].set_title('Frequency Deviation in Area 1 (ΔF₁)', fontweight='bold', fontsize=14)
    axes1[0].set_xlabel('Time (s)', fontsize=12)
    axes1[0].set_ylabel('Frequency Deviation (Hz)', fontsize=12)
    axes1[0].grid(True, alpha=0.3)
    axes1[0].legend(fontsize=12, loc='best')
    axes1[0].set_xlim(0, 10)

    axes1[1].plot(t, dF2_pso, 'g--', linewidth=2.5, label='PSO-PID', alpha=0.75)
    axes1[1].plot(t, dF2_gwo, 'b-', linewidth=2.5, label='GWO-PID', alpha=0.85)
    axes1[1].plot(t, dF2_dqn, 'r-.', linewidth=3.0, label='DQN-PID (Proposed)', alpha=0.95)
    axes1[1].set_title('Frequency Deviation in Area 2 (ΔF₂)', fontweight='bold', fontsize=14)
    axes1[1].set_xlabel('Time (s)', fontsize=12)
    axes1[1].set_ylabel('Frequency Deviation (Hz)', fontsize=12)
    axes1[1].grid(True, alpha=0.3)
    axes1[1].legend(fontsize=12, loc='best')
    axes1[1].set_xlim(0, 10)

    axes1[2].plot(t, P_tie_pso, 'g--', linewidth=2.5, label='PSO-PID', alpha=0.75)
    axes1[2].plot(t, P_tie_gwo, 'b-', linewidth=2.5, label='GWO-PID', alpha=0.85)
    axes1[2].plot(t, P_tie_dqn, 'r-.', linewidth=3.0, label='DQN-PID (Proposed)', alpha=0.95)
    axes1[2].set_title('Tie-line Power Deviation (ΔP_tie)', fontweight='bold', fontsize=14)
    axes1[2].set_xlabel('Time (s)', fontsize=12)
    axes1[2].set_ylabel('Power Deviation (p.u.MW)', fontsize=12)
    axes1[2].grid(True, alpha=0.3)
    axes1[2].legend(fontsize=12, loc='best')
    axes1[2].set_xlim(0, 10)

    plt.tight_layout()
    plt.savefig(f'{output_dir}/three_way_system_response.png', dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {output_dir}/three_way_system_response.png")
    plt.show()

    # PLOT 2: Control Effort
    fig2, axes2 = plt.subplots(2, 1, figsize=(14, 11))
    fig2.suptitle('Control Effort Comparison: DQN-PID vs GWO-PID vs PSO-PID',
                  fontsize=18, fontweight='bold')

    axes2[0].plot(t, u1_pso, 'g--', linewidth=2.5, label='PSO-PID', alpha=0.75)
    axes2[0].plot(t, u1_gwo, 'b-', linewidth=2.5, label='GWO-PID', alpha=0.85)
    axes2[0].plot(t, u1_dqn, 'r-.', linewidth=3.0, label='DQN-PID (Proposed)', alpha=0.95)
    axes2[0].set_title('Control Effort for Area 1 (u₁)', fontweight='bold', fontsize=14)
    axes2[0].set_xlabel('Time (s)', fontsize=12)
    axes2[0].set_ylabel('Control Signal (p.u.MW)', fontsize=12)
    axes2[0].grid(True, alpha=0.3)
    axes2[0].legend(fontsize=12, loc='best')
    axes2[0].set_xlim(0, 10)

    axes2[1].plot(t, u2_pso, 'g--', linewidth=2.5, label='PSO-PID', alpha=0.75)
    axes2[1].plot(t, u2_gwo, 'b-', linewidth=2.5, label='GWO-PID', alpha=0.85)
    axes2[1].plot(t, u2_dqn, 'r-.', linewidth=3.0, label='DQN-PID (Proposed)', alpha=0.95)
    axes2[1].set_title('Control Effort for Area 2 (u₂)', fontweight='bold', fontsize=14)
    axes2[1].set_xlabel('Time (s)', fontsize=12)
    axes2[1].set_ylabel('Control Signal (p.u.MW)', fontsize=12)
    axes2[1].grid(True, alpha=0.3)
    axes2[1].legend(fontsize=12, loc='best')
    axes2[1].set_xlim(0, 10)

    plt.tight_layout()
    plt.savefig(f'{output_dir}/three_way_control_effort.png', dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {output_dir}/three_way_control_effort.png")
    plt.show()

    # PLOT 3: Controller Gains
    fig3, ax3 = plt.subplots(1, 1, figsize=(14, 8))
    fig3.suptitle('Controller Gains Comparison - Three Methods', fontsize=18, fontweight='bold')

    controllers = ['PSO-PID', 'GWO-PID', 'DQN-PID\n(Proposed)']
    kp_values = [K_P1_PID_PSO, K_P1_PID_GWO, K_P1_PID_DQN]
    ki_values = [K_I1_PID_PSO, K_I1_PID_GWO, K_I1_PID_DQN]
    kd_values = [K_D1_PID_PSO, K_D1_PID_GWO, K_D1_PID_DQN]

    x = np.arange(len(controllers))
    width = 0.25

    bars1 = ax3.bar(x - width, kp_values, width, label='Kₚ', alpha=0.8,
                    color='skyblue', edgecolor='navy', linewidth=1.5)
    bars2 = ax3.bar(x, ki_values, width, label='Kᵢ', alpha=0.8,
                    color='lightgreen', edgecolor='darkgreen', linewidth=1.5)
    bars3 = ax3.bar(x + width, kd_values, width, label='Kd', alpha=0.8,
                    color='salmon', edgecolor='darkred', linewidth=1.5)

    ax3.set_title('PID Controller Gains Comparison', fontweight='bold', fontsize=16)
    ax3.set_ylabel('Gain Values', fontsize=14)
    ax3.set_xlabel('Optimization Method', fontsize=14)
    ax3.set_xticks(x)
    ax3.set_xticklabels(controllers, fontsize=13, fontweight='bold')
    ax3.legend(fontsize=14, loc='best')
    ax3.grid(True, alpha=0.3, axis='y')

    for i in range(len(controllers)):
        ax3.text(i - width, kp_values[i] + max(kp_values)*0.02, f'{kp_values[i]:.3f}',
                ha='center', va='bottom', fontweight='bold', fontsize=11)
        ax3.text(i, ki_values[i] + max(ki_values)*0.02, f'{ki_values[i]:.3f}',
                ha='center', va='bottom', fontweight='bold', fontsize=11)
        ax3.text(i + width, kd_values[i] + max(kd_values)*0.02, f'{kd_values[i]:.3f}',
                ha='center', va='bottom', fontweight='bold', fontsize=11)

    plt.tight_layout()
    plt.savefig(f'{output_dir}/three_way_gains_comparison.png', dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {output_dir}/three_way_gains_comparison.png")
    plt.show()

    # Performance metrics
    st_f1_gwo, os_f1_gwo, us_f1_gwo = compute_metrics(dF1_gwo, t)
    st_f2_gwo, os_f2_gwo, us_f2_gwo = compute_metrics(dF2_gwo, t)
    st_ptie_gwo, os_ptie_gwo, us_ptie_gwo = compute_metrics(P_tie_gwo, t)

    st_f1_pso, os_f1_pso, us_f1_pso = compute_metrics(dF1_pso, t)
    st_f2_pso, os_f2_pso, us_f2_pso = compute_metrics(dF2_pso, t)
    st_ptie_pso, os_ptie_pso, us_ptie_pso = compute_metrics(P_tie_pso, t)

    st_f1_dqn, os_f1_dqn, us_f1_dqn = compute_metrics(dF1_dqn, t)
    st_f2_dqn, os_f2_dqn, us_f2_dqn = compute_metrics(dF2_dqn, t)
    st_ptie_dqn, os_ptie_dqn, us_ptie_dqn = compute_metrics(P_tie_dqn, t)

    print("\n" + "="*120)
    print("COMPREHENSIVE PERFORMANCE COMPARISON")
    print("="*120)
    print(f"{'Controller':<15} {'Signal':<10} {'Settling (s)':<15} {'Overshoot':<15} {'Undershoot':<15} {'Cost':<12}")
    print("-"*120)

    print(f"{'PSO-PID':<15} {'ΔF₁':<10} {st_f1_pso:<15.3f} {os_f1_pso:<15.6f} {us_f1_pso:<15.6f} {pid_best_score_pso:<12.3f}")
    print(f"{'PSO-PID':<15} {'ΔF₂':<10} {st_f2_pso:<15.3f} {os_f2_pso:<15.6f} {us_f2_pso:<15.6f}")
    print(f"{'PSO-PID':<15} {'ΔP_tie':<10} {st_ptie_pso:<15.3f} {os_ptie_pso:<15.6f} {us_ptie_pso:<15.6f}")
    print("-"*120)

    print(f"{'GWO-PID':<15} {'ΔF₁':<10} {st_f1_gwo:<15.3f} {os_f1_gwo:<15.6f} {us_f1_gwo:<15.6f} {pid_best_score_gwo:<12.3f}")
    print(f"{'GWO-PID':<15} {'ΔF₂':<10} {st_f2_gwo:<15.3f} {os_f2_gwo:<15.6f} {us_f2_gwo:<15.6f}")
    print(f"{'GWO-PID':<15} {'ΔP_tie':<10} {st_ptie_gwo:<15.3f} {os_ptie_gwo:<15.6f} {us_ptie_gwo:<15.6f}")
    print("-"*120)

    print(f"{'DQN-PID':<15} {'ΔF₁':<10} {st_f1_dqn:<15.3f} {os_f1_dqn:<15.6f} {us_f1_dqn:<15.6f} {pid_best_score_dqn:<12.3f}")
    print(f"{'DQN-PID':<15} {'ΔF₂':<10} {st_f2_dqn:<15.3f} {os_f2_dqn:<15.6f} {us_f2_dqn:<15.6f}")
    print(f"{'DQN-PID':<15} {'ΔP_tie':<10} {st_ptie_dqn:<15.3f} {os_ptie_dqn:<15.6f} {us_ptie_dqn:<15.6f}")

    def improvement_percentage(dqn_val, baseline_val, lower_is_better=True):
        if baseline_val == 0: return 0
        if lower_is_better:
            return ((baseline_val - dqn_val) / baseline_val) * 100
        else:
            return ((dqn_val - baseline_val) / baseline_val) * 100

    print("\n" + "="*100)
    print("DQN-PID IMPROVEMENTS")
    print("="*100)

    print("\n--- vs PSO-PID ---")
    print(f"Cost:          {improvement_percentage(pid_best_score_dqn, pid_best_score_pso):+.2f}%")
    print(f"Settling ΔF₁:  {improvement_percentage(st_f1_dqn, st_f1_pso):+.2f}%")
    print(f"Settling ΔF₂:  {improvement_percentage(st_f2_dqn, st_f2_pso):+.2f}%")

    print("\n--- vs GWO-PID ---")
    print(f"Cost:          {improvement_percentage(pid_best_score_dqn, pid_best_score_gwo):+.2f}%")
    print(f"Settling ΔF₁:  {improvement_percentage(st_f1_dqn, st_f1_gwo):+.2f}%")
    print(f"Settling ΔF₂:  {improvement_percentage(st_f2_dqn, st_f2_gwo):+.2f}%")

    print("\n" + "="*100)
    print("RANKING")
    print("="*100)
    costs = [('PSO-PID', pid_best_score_pso), ('GWO-PID', pid_best_score_gwo), ('DQN-PID', pid_best_score_dqn)]
    costs_sorted = sorted(costs, key=lambda x: x[1])

    for rank, (method, cost) in enumerate(costs_sorted, 1):
        medal = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉"
        print(f"{medal} Rank {rank}: {method:<12} (Cost: {cost:.3f})")

    print("\n✅ Analysis Complete!")
    print(f"📊 All plots saved to: {output_dir}/")
    print("="*100)